In [1]:
import os
import json
import urllib.request

# 1. Setup local raw data directory relative to the notebooks folder
raw_dir = "../data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Define how many matches you want to download initially
MAX_MATCHES = 50 

print("Fetching competition data to locate available match IDs...")

# Target a major, high-volume league season (La Liga 2017/2018)
competitions_url = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/11/90.json"

try:
    # Read the competition match index file
    with urllib.request.urlopen(competitions_url) as response:
        match_data = json.loads(response.read().decode('utf-8'))
    
    # Extract unique match IDs
    match_ids = [str(match['match_id']) for match in match_data][:MAX_MATCHES]
    print(f"Discovered {len(match_ids)} matches available for downloading.")

    # 2. Iterative Ingestion Loop
    downloaded_count = 0
    for i, match_id in enumerate(match_ids):
        destination_path = os.path.join(raw_dir, f"{match_id}.json")
        
        # Guard clause: avoid re-downloading files that exist locally
        if os.path.exists(destination_path):
            continue
            
        event_url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id}.json"
        
        try:
            urllib.request.urlretrieve(event_url, destination_path)
            downloaded_count += 1
            if downloaded_count % 10 == 0 or i == len(match_ids) - 1:
                print(f"Progress: Checked {i+1}/{len(match_ids)} files... Downloaded {downloaded_count} new matches.")
        except Exception as file_error:
            print(f"Skipping match {match_id} due to an error: {file_error}")
            
    print(f"\n✅ Data collection complete! Local files are saved in: {raw_dir}")

except Exception as global_error:
    print(f"Could not connect to StatsBomb index repository: {global_error}")

Fetching competition data to locate available match IDs...
Discovered 35 matches available for downloading.
Progress: Checked 10/35 files... Downloaded 10 new matches.
Progress: Checked 20/35 files... Downloaded 20 new matches.
Progress: Checked 30/35 files... Downloaded 30 new matches.
Progress: Checked 35/35 files... Downloaded 35 new matches.

✅ Data collection complete! Local files are saved in: ../data/raw
